# HW2 — Part 3: Partially Observable CartPole
### POMDP · PPO under Hidden State · Observation Masking
**Yogeshvar Reddy Kallam** · IST 597 Deep RL · Penn State Spring 2025

---

## Partially Observable CartPole

Standard CartPole observation: `[cart_pos, cart_vel, pole_angle, pole_vel]`  
We **zero out velocities** → `[cart_pos, 0, pole_angle, 0]`

This creates a **POMDP** — a single frame no longer determines the system's future. Tests whether PPO can still learn despite hidden state.

In [ ]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from gymnasium.wrappers import TransformObservation
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy

def partial_obs(obs):
    obs = np.array(obs, dtype=np.float32)
    obs[1]=0.0; obs[3]=0.0   # hide velocities
    return obs

def partial_obs_space():
    return spaces.Box(
        low =np.array([-4.8,0.,-0.41887903,0.],dtype=np.float32),
        high=np.array([ 4.8,0., 0.41887903,0.],dtype=np.float32))

def make_full():    return gym.make("CartPole-v1")
def make_partial(): return TransformObservation(gym.make("CartPole-v1"), partial_obs, partial_obs_space())

# Train both
for label, make_fn, steps in [("Full obs", make_full, 150_000),
                               ("Partial obs", make_partial, 300_000)]:
    vec = DummyVecEnv([make_fn])
    m = PPO("MlpPolicy", vec, verbose=0, gamma=0.99, n_steps=2048, learning_rate=3e-4)
    m.learn(total_timesteps=steps, progress_bar=False)
    mr, sr = evaluate_policy(m, vec, n_eval_episodes=20, deterministic=True)
    print(f"{label:12s}: mean={mr:.1f} ± {sr:.1f}  (max=500)")
